# Ubuntu 파이썬 설치

- AWS Ubuntu 에는 python 이 설치 되어있다.
- 설치된 파이썬 버전 확인
  - `python3 --version`

## uv 패키지 매니저 설치

- 설치
    ```bash
    curl -Ls https://astral.sh/uv/install.sh | bash
    ```(Linux용 install 프로그램)
- uv 설치 과정에서 `.bashrc`에 PATH 설정이 추가되므로, 이미 열려 있는 셸에서는 해당 설정을 즉시 반영하기 위해 `.bashrc`를 source로 다시 로드해야 한다.
    ```bash
    source ~/.bashrc
    ```

> `.bashrc`: bash sell의 환경설정 파일

# VSCode를 이용해 EC2 instance 연결

- VSCode의 **Remote-SSH** Extension을 이용해 EC2 instance에 연결해 Local 환경처럼 개발 할 수있다.
- 프리티어 사양에서는 접속이 원활하지 않을 수있다.

1. Remote-SSH 확장 설치
    - VSCode에서 SSH를 이용해 원격으로 접속할 수있게 해주는 확장프로그램.
      
![img](figures/python/vscode1.png)

2. 왼쪽 하단에 **원격창 열기** 아이콘을 클릭한다.

![img](figures/python/vscode2.png)

3. **"Connect to Host..."/"호스트에 연결..."** 선택

![img](figures/python/vscode3.png)

4. **"SSH 호스트 구성"** 선택

![img](figures/python/vscode4.png)

5. **`사용자home\.ssh\config`** 파일선택

![img](figures/python/vscode5.png)

6. 연결설정
```bash
Host 설정이름
    HostName EC2 public ip
    User EC2 사용자(ubuntu)
    IdentityFile  ssh private key파일 경로
```
- 연결 설정을 여러개 만들 경우 위의 설정을 **아래 추가**한다.
  
![img](figures/python/vscode6.png)

7. 왼쪽 하단에 원격창 열기 다시 실행
    - **"Connect to Host..."/"호스트에 연결..."** 선택

![img](figures/python/vscode7.png)

8. `7`에서 등록한 Host 이름을 선택한다.

![img](figures/python/vscode8.png)

9. EC2 Instance의 O/S (Linux)를 선택한다.
10. `계속` 을 선택한다. 

![img](figures/python/vscode9.png)

11. 연결 된 **EC2 인스턴스에 VSCode 확장 프로그램을 설치** 해야 한다.
     - 파이썬 개발을 위해 **python**, **jupyter** 확장을 설치한다.

![img](figures/python/vscode11.png)

![img](figures/python/vscode10.png)

# EC2에 Docker 설치

- **시스템 업데이트**(새로 설치 시 필수)
    ```bash
    sudo apt update
    sudo apt upgrade
    ```

- **Docker 설치시 필요한 패키지 설치**
    ```bash
    sudo apt install -y apt-transport-https ca-certificates curl software-properties-common
    ```
-  **Docker GPG 키 추가**
    -  패키지의 출처와 무결성을 보장하기 위한 보안 서명 도구로 Docker의 패키지가 공식 버전인 것을 검증한다.
        ```bash
        curl -fsSL https://download.docker.com/linux/ubuntu/gpg | sudo gpg --dearmor -o /usr/share/keyrings/docker-archive-keyring.gpg
        ```

- **APT Repository에 Docker 저장소 추가**
    - Docker저장소는 Docker 설치와 업데이트에 필요한 패키지가 위치한 원격 서버이다.  Ubuntu의 기본 저장소에 Docker의 최신 버전이 없기 때문에 Docker의 공식 저장소를 추가해줘야 한다.

        ```bash
        echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/docker-archive-keyring.gpg] https://download.docker.com/linux/ubuntu $(lsb_release -cs) stable" | sudo tee /etc/apt/sources.list.d/docker.list > /dev/null
        ```

- **Docker Community 설치**
    ```bash
    sudo apt update
    sudo apt install -y docker-ce
    ```

-  **현재 사용자를 docker group에 추가**
    ```bash
    getent group docker            # docker 그룹이 있는지 확인
    sudo usermod -aG docker ubuntu # ubuntu 계정을 docker 그룹에 추가. usermod: 사용자 정보 변경 명령어. -aG: 사용자(ubuntu)에게 사용자 그룹(docker) 추가.(appendGroup) 
    ```
   - **SSH 연결 다시한다**.(로그아웃, 로그인) - exit or 창 끄 재연결

- **Docker Version 확인**
    ```bash
    docker --version
    ```

-  **Qdrant Image pull  및 Container 실행**
    ```bash
        sudo docker pull qdrant/qdrant
        sudo docker run -p 6333:6333 -p 6334:6334  -v "$(pwd)/qdrant_storage:/qdrant/storage:z" qdrant/qdrant &
    ```
    - 실행 명령 뒤에 &: background에서 실행

- **보안그룹에 인바운드 규칙 추가**
  - 6333, 6334 번 포트 열어 준다.
  - **EC2 > 네트워크 및 보안 > 보안그룹** 선택
    - EC2 instance가 사용중인 **보안그룹 ID** 선택
    - **인바운드 규칙**에서 **인바운드 규칙 편집** 선택 후 추가

- **Dashboard 접속**
  - `http://<<public ip>>:6333/dashboard` 로 연결
  ex. 3.36.200.116:6333/dashboard